In [40]:
import pandas as pd
import requests
import pycountry
import sqlalchemy
import psycopg
from dotenv import load_dotenv

world_bank_api_url = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&per_page=20000"

data = requests.get(world_bank_api_url).json()[1]

df = pd.json_normalize(data)

df = df[["country.value", "countryiso3code", "date", "value"]]

df = df.rename(columns={
    "country.value": "country_name",
    "countryiso3code": "country_code",
    "date": "year",
    "value": "population"
})

valid_country_codes = {country.alpha_3 for country in pycountry.countries}

inclusion = ["XKX", "CHI"]

Valid_country_codes = valid_country_codes.union(inclusion)

df = df[df["country_code"].isin(Valid_country_codes)]

In [41]:
country_names = {
    "Venezuela, RB": "Venezuela",
    "Iran, Islamic Rep.": "Iran",
    "Korea, Rep.": "South Korea",
    "Korea, Dem. People's Rep.": "North Korea",
    "Egypt, Arab Rep.": "Egypt",
    "Russian Federation": "Russia",
    "Syrian Arab Republic": "Syria",
    "Yemen, Rep.": "Yemen",
    "Viet Nam": "Vietnam",
    "Tanzania, United Republic of": "Tanzania",
    "St. Martin (French part)": "French St. Martin",
    "Sint Maarten (Dutch part)": "Dutch Sint Maarten",
    "Puerto Rico (US)": "Puerto Rico",
    "Micronesia, Fed. Sts.": "Micronesia",
    "Moldova, Republic of": "Moldova",
    "Bahamas, The": "Bahamas",
    "Virgin Islands, British": "British Virgin Islands",
    "Virgin Islands (U.S.)": "American Virgin Islands",
    "West Bank and Gaza": "Palestine",
    "Congo, Dem. Rep.": "Democratic Republic of the Congo",
    "Congo, Rep.": "Republic of the Congo",
    "Somalia, Fed. Rep.": "Somalia",
    "Slovak Republic": "Republic of Slovakia",
}

def rename_countries(country):
    
    country = str(country).strip()

    if country in country_names:
        return country_names[country]
    else:
        return country

df["country_name"] = df["country_name"].apply(rename_countries)

country_enrichment = pd.read_csv(r"C:\Users\Willi\Desktop\Data Analytic Projects\Population analysis\country_enrichment.csv")
df = df.merge(country_enrichment, on=["country_code", "country_name"], how="left")

df = df[df.isnull().any(axis=1)]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print(df[["country_name", "country_code", "land_area_km2", "capital", "continent"]].drop_duplicates().to_string(index=False))

                    country_name country_code  land_area_km2                   capital     continent
                     Afghanistan          AFG         652230                     Kabul          Asia
                         Albania          ALB          28748                    Tirana        Europe
                         Algeria          DZA        2381741                   Algiers        Africa
                  American Samoa          ASM            199                 Pago Pago       Oceania
                         Andorra          AND            468          Andorra la Vella        Europe
                          Angola          AGO        1246700                    Luanda        Africa
             Antigua and Barbuda          ATG            442              Saint John's North America
                       Argentina          ARG        2780400              Buenos Aires South America
                         Armenia          ARM          29743                   Yerevan     